In [0]:
%run ../../src/utils/config

In [0]:
%run ../../src/ingestion/pokemon_api

In [0]:
import os
from pyspark.sql import functions as F

config_path = os.path.abspath(
    os.path.join(os.getcwd(), "../../config/config.yml")
)
config = load_config(config_path)

pages = fetch_pokemon_index(
    base_url=config["source"]["base_url"],
    endpoint=config["source"]["endpoint"],
    limit=config["source"]["page_size"]
)

print(f"Index pages acquired: {len(pages)}")

pokemon_urls = extract_pokemon_urls(pages)

print(f"Pokemon detail URLs: {len(pokemon_urls)}")

In [0]:
pokemon_details = fetch_all_pokemon_details(
    pokemon_urls
)

print(f"Pokemon details acquired: {len(pokemon_details)}")

In [0]:
bronze_df = spark.createDataFrame(pokemon_details)

bronze_df = (
    bronze_df
    .withColumn(
        "_ingestion_timestamp",
        F.current_timestamp()
    )
    .withColumn(
        "_source",
        F.lit("pokeapi")
    )
    .withColumn(
        "status_code",
        F.col("status_code").cast("int")
    )
)

bronze_df.printSchema()


bronze_table = (
    f"{config['target']['catalog']}."
    f"{config['target']['bronze_schema']}."
    "pokemon_raw"
)

bronze_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable(bronze_table)

print(f"Bronze records written: {bronze_df.count()}")